# Markov Chain Monte Carlo applied to dark energy

The **Markov Chain Monte Carlo (MCMC)** methods are a family of algorithms used to sample probability distributions and evaluate uncertainties accurately. In this work these methods are used to provide estimates of the key cosmological parameters, such as the Hubble parameter and the matter and energy densities of the universe.

## Key concepts:
 - Markov chain: A process whose future depends only on the current state, not on the previous ones.
 - Metropolis–Hastings: Generates new samples through random proposals and accepts or rejects them with a probability that depends on the target distribution.

We import the required libraries.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sys
sys.path.append("src")
from mcmc import (
    lcdm_model, cpl_model, inverse_variance,
    metropolis_hastings, run_chains, gelman_rubin, running_gelman_rubin, summarize,
    best_fit, model_selection_table,
    plot_traces, plot_gelman_rubin, plot_hz_band, plot_model_comparison, create_marginal,
)

np.random.seed(42)  # reproducibility

Let's look at a plot. We now import the data.

In [ ]:
dataHz = pd.read_csv("data/hubble_cosmic_chronometers.txt", sep=' ', header=None, comment='#')

We now separate the data: **redshift**, **observations** and **errors**.

In [ ]:
redshift, obs, errors = [dataHz[i] for i in range(0,3)]

First we plot the observational data:

In [ ]:
zvals = np.linspace(0,2.2,220)
plt.errorbar(redshift,obs, yerr=errors, fmt='o',capsize=5, color='navy',ms=5, label="Obs. data")
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.grid(color='lightgray',alpha=0.5,zorder=1)

plt.ylabel("$H(z)$ [$km \\cdot s^{-1}/Mpc$]")
plt.xlabel("$z$")
plt.legend()
plt.title("Evolution of $H(z)$")
plt.savefig("figures/observational_data.png")
plt.show()

We can write $H$ using the (1st) Friedmann equation in the $\Lambda$-CDM model:
$$ H^{2} = \frac{8\pi G}{3}\rho - \frac{kc^{2}}{a^{2}} + \frac{\Lambda c^{2}}{3} $$
and defining the density parameters
$$ \rho_{c} = \frac{3H_{0}^{2}}{8\pi G} $$
$$\Omega_{m} = \frac{8\pi G}{3H_{0}^{2}}\rho_{m0} $$
$$ \Omega_{k} = \frac{-kc^{2}}{(a_{0}H_{0})^{2}}  $$
$$ \Omega_{\Lambda} = \frac{\Lambda c^{2}}{3H_{0}^{2}}$$
we can then rewrite the 1st Friedmann equation as
$$ H(z)^{2} = H_{0}^{2}\left(\Omega_{m}(1+z)^{3}+\Omega_{k}(1+z)^{2} + \Omega_{\Lambda} \right) $$
Currently, $\Omega_{k}=0$ is assumed, so the equation becomes
$$ H(z)^{2} = H_{0}^{2}\left( \Omega_{m}(1+z)^{3} + (1-\Omega_{m} ) \right) $$

## The protagonist: Metropolis–Hastings

The Metropolis–Hastings sampler now lives in the reusable package `src/mcmc/` (`sampler.py`), together with burn-in handling, acceptance-rate tracking and the **Gelman–Rubin** convergence diagnostic. Below we configure it and run **many independent chains** so convergence can be assessed rigorously.

In [ ]:
# Observed data and inverse covariance.
# Independent Gaussian errors -> C^{-1} = diag(1/sigma^2)  (the variance fix:
# chi^2 must divide by sigma^2, not sigma).
observed = (redshift.values, obs.values)
inv_cov = inverse_variance(errors.values)

In [ ]:
# MCMC configuration
N_CHAINS    = 16      # many chains -> robust Gelman-Rubin diagnostic
N_STEPS     = 15000   # steps per chain (LCDM)
BURN_IN     = 3000
N_STEPS_CPL = 40000   # CPL needs longer chains (w0-wa degeneracy)
BURN_IN_CPL = 10000
THIN        = 40      # thinning for the (KDE) marginal plots only

## $\Lambda$-CDM model

$$ H(z)^{2} = H_{0}^{2}\left(\Omega_{m}(1+z)^{3} + (1-\Omega_{m} \right)(1+z)^{3(1+w)}$$

In [ ]:
# Over-dispersed initial states; uniform-prior bounds make the posterior proper.
bounds_lcdm = [(50, 90), (0.05, 0.6)]

def init_lcdm(rng):
    return [rng.uniform(60, 80), rng.uniform(0.1, 0.5)]

proposal_std_lcdm = [0.25, 0.005]
chains_lcdm, acc_lcdm = run_chains(
    N_CHAINS, N_STEPS, init_lcdm, proposal_std_lcdm,
    lcdm_model, observed, inv_cov, burn_in=BURN_IN, bounds=bounds_lcdm, seed=42,
)
print(f"LCDM acceptance rate: mean {acc_lcdm.mean():.2f} "
      f"(min {acc_lcdm.min():.2f}, max {acc_lcdm.max():.2f})")

### Convergence diagnostics — $\Lambda$-CDM
Trace plots, running $\hat{R}$, and the posterior summary.

In [ ]:
param_names_lcdm = ["H_0", "\\Omega_m"]
plot_traces(chains_lcdm, param_names_lcdm, out="figures/traces_LCDM.png")

In [ ]:
iters_lcdm, rhat_lcdm = running_gelman_rubin(chains_lcdm)
plot_gelman_rubin(iters_lcdm, rhat_lcdm, param_names_lcdm, out="figures/gelman_rubin_LCDM.png")
print("LCDM R-hat:", dict(zip(param_names_lcdm, gelman_rubin(chains_lcdm).round(4))))

In [ ]:
pooled_lcdm = np.vstack(chains_lcdm)
mean_lcdm = pooled_lcdm.mean(axis=0)
summarize(pooled_lcdm, param_names_lcdm)

### Posterior fit — $\Lambda$-CDM

In [ ]:
zline = np.arange(0, 2, 0.1)
plt.errorbar(redshift, obs, yerr=errors, fmt='o', capsize=5, color='#021526', ms=5, label="Obs. data")
plt.plot(zline, lcdm_model(mean_lcdm)(zline), color='#03346E', label=r"$\Lambda$-CDM (posterior mean)")
plt.text(0.05, 0.9, rf"$H_0$: {mean_lcdm[0]:.2f} km/s/Mpc,  $\Omega_m$: {mean_lcdm[1]:.3f}",
         transform=plt.gca().transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', edgecolor='black', facecolor='white'))
plt.ylabel("$H(z)$ [$km \\cdot s^{-1}/Mpc$]"); plt.xlabel("$z$")
plt.title("Evolution of $H(z)$"); plt.grid(True); plt.legend()
plt.savefig("figures/observational_data_LCDM.png"); plt.show()

In [ ]:
plot_hz_band(redshift.values, obs.values, errors.values, lcdm_model, pooled_lcdm,
             out="figures/LCDM_confidence_intervals.png",
             title="$\Lambda$-CDM: $H(z)$ with posterior confidence intervals")

In [ ]:
H0_lcdm, Om_lcdm = pooled_lcdm[::THIN, 0], pooled_lcdm[::THIN, 1]

In [ ]:
create_marginal(H0_lcdm, Om_lcdm, "H_0 [km \\cdot s^{-1}/Mpc]", "\\Omega_{m}")

## Alternatives to the $\Lambda$-CDM model

An alternative to modelling dark energy as a cosmological constant comes from the equation of state in cosmology:
$$ \rho = \frac{p}{\rho} $$

$$w = \frac{p}{\rho} $$

### CPL model

$$ H(z)^{2} = H_{0}^{2}\left(\Omega_m \cdot (1 + z)^3 + (1 - \Omega_m) \cdot (1 + z)^{3(1 + w_0 + w_{a}) \cdot e^{-3w_{a} \left(\frac{z}{z+1}\right)}} \right)$$

In [ ]:
# CPL has a strong w0-wa degeneracy, so a diagonal random walk mixes poorly.
# Strategy: a short PILOT run estimates the posterior covariance, which is then
# used as a preconditioned proposal (2.38^2/d optimal scaling) for the main
# chains. (Low acceptance with such informed steps is expected and fine.)
bounds_cpl = [(50, 90), (0.05, 0.6), (-2.0, 0.0), (-3.0, 3.0)]

pilot_cpl, _ = metropolis_hastings(
    10000, [70, 0.3, -0.9, 0.0], [0.3, 0.006, 0.06, 0.15],
    cpl_model, observed, inv_cov, burn_in=2000, bounds=bounds_cpl, rng=0,
)
proposal_cov_cpl = np.cov(pilot_cpl.T)  # preconditioner from the pilot

def init_cpl(rng):
    return [rng.uniform(65, 75), rng.uniform(0.2, 0.4),
            rng.uniform(-1.1, -0.7), rng.uniform(-0.3, 0.3)]

chains_cpl, acc_cpl = run_chains(
    N_CHAINS, N_STEPS_CPL, init_cpl, proposal_cov_cpl,
    cpl_model, observed, inv_cov, burn_in=BURN_IN_CPL, bounds=bounds_cpl, seed=7,
)
print(f"CPL acceptance rate: mean {acc_cpl.mean():.3f} "
      f"(preconditioned proposal; low value expected)")

### Convergence diagnostics — CPL

In [ ]:
param_names_cpl = ["H_0", "\\Omega_m", "w_0", "w_a"]
plot_traces(chains_cpl, param_names_cpl, out="figures/traces_CPL.png")

In [ ]:
iters_cpl, rhat_cpl = running_gelman_rubin(chains_cpl)
plot_gelman_rubin(iters_cpl, rhat_cpl, param_names_cpl, out="figures/gelman_rubin_CPL.png")
print("CPL R-hat:", dict(zip(param_names_cpl, gelman_rubin(chains_cpl).round(4))))

In [ ]:
pooled_cpl = np.vstack(chains_cpl)
mean_cpl = pooled_cpl.mean(axis=0)
summarize(pooled_cpl, param_names_cpl)

### Model comparison

In [ ]:
zline = np.arange(0, 2, 0.1)
plt.errorbar(redshift, obs, yerr=errors, fmt='o', capsize=5, label="Obs. data", color="#0C134F")
plt.plot(zline, cpl_model(mean_cpl)(zline), color='#1D267D', linestyle="--", label=r"CPL (posterior mean)")
plt.plot(zline, lcdm_model(mean_lcdm)(zline), color='#D4ADFC', label=r"$\Lambda$-CDM (posterior mean)")
plt.text(0.03, 0.92, rf"CPL: $H_0$={mean_cpl[0]:.1f}, $\Omega_m$={mean_cpl[1]:.3f}, $w_0$={mean_cpl[2]:.2f}, $w_a$={mean_cpl[3]:.2f}",
         transform=plt.gca().transAxes, fontsize=7, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', edgecolor='black', facecolor='white'))
plt.text(0.03, 0.82, rf"$\Lambda$-CDM: $H_0$={mean_lcdm[0]:.1f}, $\Omega_m$={mean_lcdm[1]:.3f}",
         transform=plt.gca().transAxes, fontsize=7, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', edgecolor='black', facecolor='white'))
plt.xlabel("z"); plt.ylabel("$H(z)$ [$km \\cdot s^{-1}/Mpc$]")
plt.title("Evolution of $H(z)$"); plt.legend(); plt.grid(True)
plt.savefig("figures/LCDM_CPL_comparison.png"); plt.show()

In [ ]:
H0_cpl, Om_cpl, w0_cpl, wa_cpl = [pooled_cpl[::THIN, i] for i in range(4)]

In [ ]:
create_marginal(H0_cpl, Om_cpl, "H_0 [km \\cdot s^{-1}/Mpc]", "\\Omega_{m}")

In [ ]:
create_marginal(w0_cpl, wa_cpl, "w_0", "w_a")

## Model selection — does the data justify dynamical dark energy?

CPL will always fit at least as well as ΛCDM because it *contains* ΛCDM (at
$w_0=-1,\, w_a=0$), so a lower $\chi^2$ alone proves nothing. To decide whether
the two extra parameters are **warranted**, we penalize complexity with:

- **reduced** $\chi^2/\mathrm{dof}$ — goodness of fit per degree of freedom;
- **AIC** $= \chi^2_{\min} + 2k$ — mild complexity penalty;
- **BIC** $= \chi^2_{\min} + k\ln n$ — stronger penalty (here $n=36$).

Lower is better. A model is disfavoured by roughly $\Delta>2$ (positive),
$\Delta>6$ (strong), $\Delta>10$ (very strong).

In [ ]:
n = len(observed[1])

state_lcdm, chi2_lcdm = best_fit(lcdm_model, mean_lcdm, observed, inv_cov, bounds=bounds_lcdm)
state_cpl,  chi2_cpl  = best_fit(cpl_model,  mean_cpl,  observed, inv_cov, bounds=bounds_cpl)

selection = model_selection_table(
    {"ΛCDM": (state_lcdm, chi2_lcdm, 2), "CPL": (state_cpl, chi2_cpl, 4)}, n
)
selection.round(3)

In [ ]:
plot_model_comparison(selection, out="figures/model_selection.png")

### Conclusion

Both models fit well ($\chi^2/\mathrm{dof} < 1$, hinting the quoted errors are
slightly conservative). CPL lowers $\chi^2_{\min}$ by only $\approx 3$ — less than
what its two extra parameters "cost": **$\Delta\mathrm{AIC}\approx 1$** (inconclusive)
and **$\Delta\mathrm{BIC}\approx 4$** (positive evidence *for* ΛCDM).

**Does the data justify dynamical dark energy? No.** With these ~36 cosmic-chronometer
$H(z)$ points, a cosmological constant is preferred; the CPL equation-of-state parameters
$(w_0, w_a)$ are essentially unconstrained (recall the near-flat, degenerate posterior and
the extra sampling effort it required).

This sits inside the current **$H_0$ tension**: our ΛCDM posterior gives
$H_0 \approx 71.9 \pm 1.4$ km/s/Mpc, between the Planck CMB value ($\approx 67.4$) and the
local SH0ES measurement ($\approx 73$). Recent DESI BAO results (2024–2025) have revived
interest in *evolving* dark energy as a possible resolution, but breaking the $w_0$–$w_a$
degeneracy requires combining probes (BAO + CMB + SNe) — a single $H(z)$ compilation cannot
settle it.

## References

Yu, H., Ratra, B., & Wang, F. Y. (2018). Hubble parameter and Baryon Acoustic Oscillation measurement constraints on the Hubble constant, the deviation from the spatially flat ΛCDM model, the deceleration–acceleration transition redshift, and spatial curvature. The Astrophysical Journal, 856(1), 3.

Bengaly, C., Dantas, M. A., Casarini, L., & Alcaniz, J. (2023). Measuring the Hubble constant with cosmic chronometers: a machine learning approach. The European Physical Journal C, 83(6), 1-13.

Gómez-Vargas, I., Medel-Esquivel, R., García-Salcedo, R., & Vázquez, J. A. (2023). Neural network reconstructions for the Hubble parameter, growth rate and distance modulus. The European Physical Journal C, 83(4), 304.